In [21]:
%pip install pdfplumber sentence-transformers chromadb tf-keras torch -q

Note: you may need to restart the kernel to use updated packages.


In [22]:
import pdfplumber 
from sentence_transformers import SentenceTransformer
import chromadb
import os

# Phase 1 Configuration
CONFIG = {
    "model_name": "all-MiniLM-L6-v2",
    "chunk_size": 300,
    "collection_name": "pdf_collection",
    "top_k_results": 3
}

# Initialize Model
model = SentenceTransformer(CONFIG["model_name"])
print("✓ Model loaded successfully")

✓ Model loaded successfully


In [23]:
def extract_text(pdf_path):
    """
    Extract text from PDF file.
    
    Args:
        pdf_path: Path to PDF file
        
    Returns:
        Extracted text string, or None if error
    """
    text = ""
    try:
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"PDF file not found: {pdf_path}")
            
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            for i, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
            print(f"✓ Extracted text from {total_pages} pages ({len(text)} characters)")
        return text
    except Exception as e:
        print(f"✗ Error extracting PDF: {str(e)}")
        return None

In [24]:
def chunk_text(text, chunk_size=None):
    """
    Split text into chunks of specified size.
    
    Args:
        text: Input text to chunk
        chunk_size: Number of words per chunk (default: from CONFIG)
        
    Returns:
        List of text chunks
    """
    if chunk_size is None:
        chunk_size = CONFIG["chunk_size"]
        
    if not text or not text.strip():
        print("✗ Empty text provided")
        return []
    
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    
    print(f"✓ Created {len(chunks)} chunks (size: {chunk_size} words)")
    return chunks

In [25]:
def create_embeddings(chunks):
    """
    Generate embeddings for text chunks.
    
    Args:
        chunks: List of text chunks
        
    Returns:
        Array of embeddings or None if error
    """
    try:
        if not chunks:
            print("✗ No chunks provided")
            return None
            
        embeddings = model.encode(chunks, show_progress_bar=False)
        print(f"✓ Generated {len(embeddings)} embeddings (dim: {len(embeddings[0])})")
        return embeddings
    except Exception as e:
        print(f"✗ Error creating embeddings: {str(e)}")
        return None

In [26]:
def store_in_chromadb(chunks, embeddings):
    """
    Store chunks and embeddings in ChromaDB.
    
    Args:
        chunks: List of text chunks
        embeddings: Array of embeddings
        
    Returns:
        ChromaDB collection or None if error
    """
    try:
        if chunks is None or embeddings is None:
            print("✗ Chunks or embeddings are None")
            return None
            
        if len(chunks) != len(embeddings):
            print("✗ Chunk and embedding count mismatch")
            return None
        
        client = chromadb.Client()
        
        # Clear existing collection if it exists
        try:
            client.delete_collection(name=CONFIG["collection_name"])
        except:
            pass
        
        collection = client.create_collection(name=CONFIG["collection_name"])

        for i, chunk in enumerate(chunks):
            collection.add(
                documents=[chunk],
                embeddings=[embeddings[i].tolist()] if hasattr(embeddings[i], 'tolist') else [embeddings[i]],
                ids=[str(i)]
            )

        print(f"✓ Stored {len(chunks)} chunks in ChromaDB")
        return collection
    except Exception as e:
        print(f"✗ Error storing in ChromaDB: {str(e)}")
        return None

In [27]:
def search(collection, query, top_k=None):
    """
    Perform semantic search on collection.
    
    Args:
        collection: ChromaDB collection
        query: Search query string
        top_k: Number of results to return (default: from CONFIG)
        
    Returns:
        List of most relevant documents
    """
    try:
        if top_k is None:
            top_k = CONFIG["top_k_results"]
            
        if not query or not query.strip():
            print("✗ Empty query")
            return []
            
        query_embedding = model.encode(query).tolist()

        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )

        return results['documents'][0] if results['documents'] else []
    except Exception as e:
        print(f"✗ Error searching: {str(e)}")
        return []

#### ML PHASE 2:

In [ ]:
#model setup
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3775.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
#PDF To Text
from PyPDF2 import PdfReader

def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

In [7]:
#chunking
def chunk_text(text, chunk_size=200):
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        
    return chunks

In [8]:
#embedding
def create_embeddings(chunks):
    embeddings = model.encode(chunks)
    return embeddings

In [10]:
#storing in chromadb
import chromadb

client = chromadb.Client()
collection = client.get_collection("ml_data")

def store_embeddings(chunks, embeddings):
    for i in range(len(chunks)):
        collection.add(
            documents=[chunks[i]],
            embeddings=[embeddings[i]],
            ids=[str(i)]
        )

In [11]:
#semantic search
def search(query):
    query_embedding = model.encode([query])
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3
    )
    
    return results['documents']

In [30]:
from docx import Document

def extract_text(docx_path):
    doc = Document(docx_path)
    text = ""
    for para in doc.paragraphs:
        text += para.text + "\n"
    return text

In [36]:
docx_text = extract_text(r"C:\Users\HP\OneDrive\Documents\sample.docx")

chunks = chunk_text(docx_text)

embeddings = create_embeddings(chunks)

store_embeddings(chunks, embeddings)

print(generate_answer("What is AI?"))

data. Underfitting When a model is too simple and cannot learn patterns properly. 5. Applications of Machine Learning Machine learning is used in many fields: Healthcare (disease prediction) Finance (fraud detection) Social media (recommendation systems) Transportation (self-driving cars) Education (smart learning systems) 6. Why Machine Learning is Important Automates decision-making Improves accuracy over time Handles large amounts of data Used in almost every modern technology Conclusion Machine learning is a powerful technology that allows systems to learn from data and improve automatically. It is the foundation of modern AI applications and is widely used in real-world problems. This is dummy data for testing classification (spam / not spam) Common Algorithms: Linear Regression Logistic Regression Decision Trees Support Vector Machines (b) Unsupervised Learning In unsupervised learning, the model is given unlabeled data and it finds patterns on its own. Example: Customer grouping

#### ML PHASE 3

In [37]:
#Query to answer
def generate_answer(query):
    context = search(query)
    
    # Combine top results
    context_text = " ".join(context[0])
    
    return context_text

#### Phase Four

In [38]:
#improve chunking
def chunk_text(text, chunk_size=150, overlap=50):
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        
    return chunks

In [39]:
#better retrieval
def search(query):
    query_embedding = model.encode([query])
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=5   # increase results
    )
    
    return results['documents']

In [40]:
#clean output
def generate_answer(query):
    context = search(query)
    
    unique_text = list(set(context[0]))
    return " ".join(unique_text)

In [ ]:

import numpy as np
def generate_answer(query, index, chunks):
    query_vec = model.encode([query])

    D, I = index.search(np.array(query_vec), k=3)

    results = [chunks[i] for i in I[0]]

    return "\n".join(results)

In [55]:

#RUN PIPELINE


file_path = r"C:\Users\HP\OneDrive\Documents\sample.docx"

# Step 1: extract text
text = extract_text(file_path)

# Step 2: chunk
chunks = chunk_text(text)

# Step 3: embeddings
embeddings = create_embeddings(chunks)

# Step 4: store in FAISS (index create yahan hoga)
index, stored_chunks = store_embeddings(chunks, embeddings)

# Step 5: query
answer = generate_answer("What is AI?", index, stored_chunks)

print("\n===== ANSWER =====\n")
print(answer)


===== ANSWER =====

data. Underfitting When a model is too simple and cannot learn patterns properly. 5. Applications of Machine Learning Machine learning is used in many fields: Healthcare (disease prediction) Finance (fraud detection) Social media (recommendation systems) Transportation (self-driving cars) Education (smart learning systems) 6. Why Machine Learning is Important Automates decision-making Improves accuracy over time Handles large amounts of data Used in almost every modern technology Conclusion Machine learning is a powerful technology that allows systems to learn from data and improve automatically. It is the foundation of modern AI applications and is widely used in real-world problems.
classification (spam / not spam) Common Algorithms: Linear Regression Logistic Regression Decision Trees Support Vector Machines (b) Unsupervised Learning In unsupervised learning, the model is given unlabeled data and it finds patterns on its own. Example: Customer grouping (clusteri